In [0]:
%run ../src/transformations

In [0]:
ORDERS_SCHEMA = ("order_id int, customer_id int, product string, quantity int, "
                 "unit_price double, discount_pct double, order_date string, "
                 "status string, updated_at string")
CUST_SCHEMA = "customer_id int, customer_name string, country string"

def sample_orders():
    return spark.createDataFrame([
        (1, 101, " Laptop ", 1, 1000.0, 10.0, "2026-09-05", "completed", "2026-09-05 10:00:00"),
        (2, 102, "Mouse", 2, 25.0, None, "2026-09-12", "COMPLETED", "2026-09-12 09:00:00"),
        (2, 102, "Mouse", 3, 25.0, None, "2026-09-12", "COMPLETED", "2026-09-13 09:00:00"),
        (3, 103, "Keyboard", 0, 50.0, 0.0, "2026-09-20", "completed", "2026-09-20 10:00:00"),
        (4, 999, "Monitor", 1, 300.0, 5.0, "2026-10-02", "shipped", "2026-10-02 10:00:00"),
        (5, 101, "Phone", 1, 500.0, 0.0, "2026-10-03", "cancelled", "2026-10-03 10:00:00"),
        (None, 101, "Cable", 1, 10.0, 0.0, "2026-10-04", "completed", "2026-10-04 10:00:00"),
    ], ORDERS_SCHEMA)

def sample_customers():
    return spark.createDataFrame(
        [(101, "Asha", "IN"), (102, "Ben", "US"), (102, "Ben", "US")], CUST_SCHEMA)

def row(df, oid):
    return df.filter(f"order_id = {oid}").collect()[0]

def test_clean_drops_null_id_and_zero_qty():
    ids = {r.order_id for r in clean_orders(sample_orders()).collect()}
    assert ids == {1, 2, 4, 5}, ids

def test_clean_keeps_latest_duplicate():
    out = clean_orders(sample_orders())
    assert out.filter("order_id = 2").count() == 1
    assert row(out, 2).quantity == 3

def test_clean_trims_and_uppercases():
    r = row(clean_orders(sample_orders()), 1)
    assert r.product == "Laptop" and r.status == "COMPLETED"

def test_line_total_with_discount():
    r = row(add_line_total(clean_orders(sample_orders())), 1)
    assert abs(r.line_total - 900.0) < 0.01, r.line_total

def test_line_total_null_discount_is_zero():
    df = spark.createDataFrame(
        [(9, 1, "X", 4, 10.0, None, "2026-09-01", "COMPLETED", "2026-09-01 00:00:00")],
        ORDERS_SCHEMA)
    assert abs(row(add_line_total(df), 9).line_total - 40.0) < 0.01

def test_join_unknown_customer_defaults():
    out = join_customers(clean_orders(sample_orders()), sample_customers())
    r = row(out, 4)
    assert r.country == "UNKNOWN" and r.customer_name == "UNKNOWN"

def test_join_does_not_multiply_rows():
    o = clean_orders(sample_orders())
    assert join_customers(o, sample_customers()).count() == o.count()

def test_monthly_revenue():
    df = join_customers(add_line_total(clean_orders(sample_orders())), sample_customers())
    got = {(r.order_month, r.country): (r.revenue, r.order_count)
           for r in monthly_revenue(df).collect()}
    exp = {("2026-09", "IN"): (900.0, 1), ("2026-09", "US"): (75.0, 1),
           ("2026-10", "UNKNOWN"): (285.0, 1)}
    assert got.keys() == exp.keys(), got
    for k in exp:
        assert abs(got[k][0] - exp[k][0]) < 0.01 and got[k][1] == exp[k][1], (k, got[k])

tests = [v for k, v in sorted(globals().items()) if k.startswith("test_") and callable(v)]
failures = []
for t in tests:
    try:
        t()
        print(f"PASS  {t.__name__}")
    except Exception as e:
        failures.append(t.__name__)
        print(f"FAIL  {t.__name__}: {e}")

if failures:
    raise Exception(f"{len(failures)} test(s) failed: {failures}")
print(f"All {len(tests)} tests passed")

In [0]:
%skip
import pytest
from sample_data import get_orders, get_customers, ORDERS_SCHEMA
from transformations import clean_orders, add_line_total, join_customers, monthly_revenue

def by_id(df, oid):
    return df.filter(f"order_id = {oid}").collect()[0]

# clean_orders
def test_clean_removes_null_ids_and_invalid_qty(spark):
    ids = {r.order_id for r in clean_orders(get_orders(spark)).collect()}
    assert ids == {1, 2, 4, 5}          # 3 (qty 0) and null id are dropped

def test_clean_keeps_latest_duplicate(spark):
    out = clean_orders(get_orders(spark))
    assert out.filter("order_id = 2").count() == 1
    assert by_id(out, 2).quantity == 3

def test_clean_trims_and_uppercases(spark):
    row = by_id(clean_orders(get_orders(spark)), 1)
    assert row.product == "Laptop"
    assert row.status == "COMPLETED"

def test_clean_parses_dates(spark):
    assert str(by_id(clean_orders(get_orders(spark)), 1).order_date) == "2026-09-05"

# add_line_total
def test_line_total_with_discount(spark):
    out = add_line_total(clean_orders(get_orders(spark)))
    assert by_id(out, 1).line_total == pytest.approx(900.0)

def test_line_total_null_discount_is_zero(spark):
    df = spark.createDataFrame(
        [(9, 1, "X", 4, 10.0, None, "2026-09-01", "COMPLETED", "2026-09-01 00:00:00")],
        ORDERS_SCHEMA)
    assert by_id(add_line_total(df), 9).line_total == pytest.approx(40.0)

# join_customers
def test_join_unknown_customer_defaults(spark):
    out = join_customers(clean_orders(get_orders(spark)), get_customers(spark))
    row = by_id(out, 4)
    assert row.country == "UNKNOWN" and row.customer_name == "UNKNOWN"

def test_join_does_not_multiply_rows(spark):
    orders = clean_orders(get_orders(spark))
    assert join_customers(orders, get_customers(spark)).count() == orders.count()

# monthly_revenue
def test_revenue_excludes_cancelled_and_aggregates(spark):
    df = join_customers(add_line_total(clean_orders(get_orders(spark))), get_customers(spark))
    got = {(r.order_month, r.country): (r.revenue, r.order_count)
           for r in monthly_revenue(df).collect()}
    assert got == {
        ("2026-09", "IN"): (pytest.approx(900.0), 1),
        ("2026-09", "US"): (pytest.approx(75.0), 1),
        ("2026-10", "UNKNOWN"): (pytest.approx(285.0), 1),   # cancelled order 5 excluded
    }